In [5]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras .preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [6]:
vocab_size=10000
(X_train, y_train), (X_test,y_test)=imdb.load_data(
    num_words=vocab_size
)
print("Training sample:",len(X_train))
print("Testing samples:",len(X_test))

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training sample: 25000
Testing samples: 25000


In [7]:
max_length=200
X_train=pad_sequences(
    X_train,
    maxlen=max_length,
    padding='post'
)
X_test=pad_sequences(
    X_test,
    maxlen=max_length,
    padding='post'
)

In [8]:
model=Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        input_length=max_length
    ),
    SimpleRNN(64),
    Dense(32,activation='relu'),
    Dense(1,activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [10]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [11]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [12]:
history=model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=64
)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 29s 83ms/step - accuracy: 0.5177 - loss: 0.6915 - val_accuracy: 0.5080 - val_loss: 0.7010
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 41s 83ms/step - accuracy: 0.5847 - loss: 0.6591 - val_accuracy: 0.5612 - val_loss: 0.6613
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 41s 84ms/step - accuracy: 0.6263 - loss: 0.6032 - val_accuracy: 0.5334 - val_loss: 0.6884
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 28s 91ms/step - accuracy: 0.6341 - loss: 0.5835 - val_accuracy: 0.6416 - val_loss: 0.6109
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 30s 95ms/step - accuracy: 0.6689 - loss: 0.5355 - val_accuracy: 0.6160 - val_loss: 0.6241


In [13]:
loss,accuracy=model.evaluate(
    X_test,
    y_test
)
print("Accuracy:",accuracy)

782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - accuracy: 0.6005 - loss: 0.6363
Accuracy: 0.6004800200462341


In [14]:
from tensorflow.keras.datasets import imdb
word_index=imdb.get_word_index()

word_index={k:(v+3) for k,v in word_index.items()}

word_index["<PAD>"]=0
word_index["<START>"]=1
word_index["<UNK>"]=2
word_index["<UNUSED>"]=3

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [15]:
def review_to_sequence(review):
  review=review.lower().split()
  sequence=[]
  for word in review:
    sequence.append(word_index.get(word,2))
  return sequence


In [16]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

max_length=200

def predict_sentiment(review):
  sequence=review_to_sequence(review)

  padded=pad_sequences(
      [sequence],
      maxlen=max_length,
      padding='post'
  )
  prediction=model.predict(padded,verbose=0)
  score=prediction[0][0]
  print("Sentiment Score:",score)
  if score>0.5:
    print("Positive Review")
  else:
    print("Negative Review")

In [19]:
predict_sentiment("This is a good movie")

Sentiment Score: 0.46282238
Negative Review


In [18]:
predict_sentiment("This is a bad movie")

Sentiment Score: 0.4620534
Negative Review
